In [1]:
import time
start_time = time.perf_counter()
print('Done!')

Done!


In [2]:
import re
from collections import deque

the_directory = 'output/demo'
ending = 'XXXTHISENDSHEREXXX'
md_file    = 'the_start_markdown.txt'
src_file   = 'the_functions_all.txt'
out_file   = 'in_between_list.txt'

# 1) Parse Markdown tree jadi list pasangan (parent, child)
pairs = []
stack = []
with open(f'{the_directory}/{md_file}') as f:
    for line in f:
        if not line.strip(): continue
        indent = len(line) - len(line.lstrip(' '))
        func   = line.lstrip(' -').strip()
        while stack and stack[-1][0] >= indent:
            stack.pop()
        if stack:
            pairs.append((stack[-1][1], func))
        stack.append((indent, func))

# 2) Load semua blok source dan build adjacency list
with open(f'{the_directory}/{src_file}') as f:
    content = f.read()

pattern = rf"Source Code for\s+([\w_]+)\s*:\s*\n(.*?)(?={re.escape(ending)})"
blocks  = dict(re.findall(pattern, content, flags=re.DOTALL))
adj     = {fn: set() for fn in blocks}

for fn, src in blocks.items():
    for callee in blocks:
        if callee != fn and re.search(rf'\b{re.escape(callee)}\b', src):
            adj[fn].add(callee)

# 3) BFS tiap pasangan
results = []
for parent, child in pairs:
    if parent not in adj or child not in blocks:
        results.append([parent, f"(no source for {parent} or {child})"])
        continue
    # direct call?
    if child in adj[parent]:
        results.append([parent, child])
        continue

    # BFS multi-hop
    visited = {parent}
    queue   = deque([[parent]])
    found   = None
    while queue and not found:
        path = queue.popleft()
        last = path[-1]
        for nb in adj[last]:
            if nb in visited: 
                continue
            visited.add(nb)
            new_path = path + [nb]
            if nb == child:
                found = new_path
                break
            queue.append(new_path)
    if found:
        results.append(found)
    else:
        results.append([parent, f"(no path to {child})"])

# 4) Tulis hasil
with open(f"{the_directory}/{out_file}", 'w') as f:
    for chain in results:
        f.write(', '.join(chain) + '\n')

print('Done!')


Done!


In [3]:
end_time = time.perf_counter()
elapsed = end_time - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Elapsed time: {hours}h {minutes}m {seconds:.6f}s")
print('Done!')

Elapsed time: 0h 1m 42.325754s
Done!
